# Plot Observation

This notebook loads and plots data taken with the `observe()` function of `pocket_correlator.py`, which is used to record correlated data for a specified integration time until the script is stopped by the user. This software backend has been primarily used on the Deep Dish Development Array (D3A), a prototype for technology development for the Canadian Hydrogen Observatory and Radio transient Detector (CHORD) under construction at the Dominion Radio Astrophysical Observatory (DRAO) in Kaleden, British Columbia, Canada.

You can use `test_pocket_correlator_observe.py` to take data with the `observe()` method. `plot_observation` uses the helper script `load_acq_rfsoc.py`, which processes HDF5 files written by `observe()`. 

Revisions:
- **October 13, 2025**: data was taken at DRAO while D3A was down. Two 50-$\Omega$ terminations were separately amplified through 3 CHORD Stage-2 FLAs, and injected into ADCs 1 and 2. 10 s integrations were recorded. The plots checked in with this commit correspond to this signal chain.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import os
from load_acq_rfsoc import load_data

# Define Acquisition and Universal Variables

In [ ]:
# Define the acquisition
#acq = '20241217T100454Z_D3A_rfsoc'
#acq = '20241219T115750Z_D3A_rfsoc'
# acq = '20241219T145139Z_D3A_rfsoc'
# acq = '20250311T180505Z_D3A_rfsoc'
# acq = '20250519T020504Z_D3A_rfsoc'
acq = '20251013T171558Z_pathfinder_test'

# Specify the path to the data
# data_path = '/home/chiveremote/D3A_acq/data/' # chive, main data location
data_path = '/home/chiveremote/test_pc_observe_data' # chive, pathfinder test data location
# data_path = '/home/chiveremote/ih/data/' # chive, data copied to my directory
# data_path = '/home/ianhendr/projects/rrg-kmsmith/d3a/data' # Cedar

# Specify the path to the gain file. Make sure you load the gains associated with the corresponding acquisition!
# gains_path = f'/home/chiveremote/D3A_acq/data/digital_gains/{acq}_digitalgain' # chive, main data location
# gains_path = f'/home/chiveremote/ih/data/digital_gains/{acq}_digitalgain' # chive, data copied to my directory
gains_path = f'{data_path}/digital_gains/{acq}_digitalgain'

# Gather Data, Gains, and Statistics

In [ ]:
len(os.listdir(data_path + '/' + acq))

In [ ]:
file_names = os.listdir(f'{data_path}/{acq}')
N_FILES = len(file_names)

np.sort([int(os.listdir(data_path + '/' + acq)[n].split('.hdf5')[0]) for n in range(len(os.listdir(data_path + '/' + acq)))])
file_numbers = np.sort([int(file_name.split('.hdf5')[0]) for file_name in file_names])
file_numbers

## Data and Statistics

In [ ]:
# If the acq is earlier than March 11, 2025, use the following:
# vis, f, prod, t, counts, glin, glog, sat, adc_overflows, fft_overflows, scaler_overflows, corr_overrun, frame_count = load_data(acq, 
#                                                                                                                                 data_path, 
#                                                                                                                                 gains_path, 
#                                                                                                                                 file_numbers = None, # np.arange(2),
#                                                                                                                                 trim_data = True,
#                                                                                                                                 apply_counts = True,
#                                                                                                                                 apply_gains = True
#                                                                                                                                 )

# If the acq is later than or equal to March 11, 2025, use the following:
vis, f, prod, t, counts, glin, glog, sat, adc_overrange, adc_overvoltage, fft_overflows, scaler_overflows, scaler_stats_ready, corr_overrun, frame_count = load_data(acq, 
                                                                                                                                                                     data_path, 
                                                                                                                                                                     gains_path, 
                                                                                                                                                                     file_numbers = None,
                                                                                                                                                                     trim_data = True,
                                                                                                                                                                     apply_counts = True,
                                                                                                                                                                     apply_gains = True
                                                                                                                                                                     )

In [ ]:
counts

In [ ]:
vis.shape

In [ ]:
N_CHAN = int(2**14 / 2) - 4
N_POLS = 8
N_PRODS = int(N_POLS*(N_POLS + 1)/2)

In [ ]:
# Generate auto prod array
auto_prod_idx = np.zeros(8)
ctr = 0
for n in range(len(prod)):
    if prod[n][0] == prod[n][1]:
        auto_prod_idx[ctr] = n
        ctr += 1

auto_prod_idx = auto_prod_idx.astype(int)
auto_prod_idx

In [ ]:
board_inputs = [f'ADC{n + 1}' for n in range(8)]

### Check out raw visibilities

In [ ]:
plt.rcParams.update({'font.size': 12})

In [ ]:
pol = 0

plt.figure(figsize = (12, 6))
plt.title(f'autocorrelation, {board_inputs[pol]}, first integration')
plt.semilogy(f, np.abs(vis[0, :, auto_prod_idx[pol]]))
plt.xticks(np.arange(0, 1600 + 100, 100))
plt.xlim(0, 1600)
# plt.xlim(1400, 1425)
plt.ylabel('visibility [ADU$^2$]')
plt.xlabel('frequency [MHz]')
plt.grid()
plt.tight_layout()

In [ ]:
pol = 0

plt.figure(figsize = (12, 6))
plt.title(f'autocorrelation, {board_inputs[pol]}')
plt.pcolormesh(f, t - t[0], 10*np.log10(np.abs(vis[..., auto_prod_idx[pol]])))
plt.colorbar(label = 'power [arb. dB]')
plt.ylabel('time [s]')
plt.gca().invert_yaxis()
plt.xlabel('frequency [MHz]')

plt.tight_layout()

In [ ]:
# Plot means of autocorrelation products over time

plt.figure(figsize = (12, 12))
plt.suptitle(f'autocorrelations, first integration')

for n in range(N_POLS):
    plt.subplot(4, 2, n + 1)
    plt.title(board_inputs[n])
    plt.semilogy(f, np.abs(vis[0, :, auto_prod_idx[n]]))

    if n in [0, 2, 4, 6]:
        plt.ylabel('visibility, [ADU$^2$]')
    if n in [6, 7]:
        plt.xlabel('frequency [MHz]')

    plt.xticks(np.arange(0, 1600 + 200, 200)) # , rotation = 45)
    plt.xlim(0, 1600)
    plt.grid()

plt.tight_layout()

## Gains

In [ ]:
plt.figure(figsize = (12, 12))

for n in range(N_POLS):
    plt.subplot(4, 2, n + 1)
    plt.title(board_inputs[n])
    plt.plot(f, glin[n]*2**glog[n])

    if n in [0, 2, 4, 6]:
        plt.ylabel('gain')
    if n in [6, 7]:
        plt.xlabel('frequency [MHz]')

    plt.xticks(np.arange(0, 1600 + 200, 200)) # , rotation = 45)
    plt.xlim(0, 1600)
    plt.grid()

plt.tight_layout()

## Investigate Statistics

In [ ]:
plt.figure(figsize = (12, 6))
plt.suptitle('ADC Overrange and Overvoltage')
for n in range(N_POLS):
    plt.subplot(211)
    plt.plot((t - t[0]), adc_overrange[:, n], label = board_inputs[n])
    plt.subplot(212)
    plt.plot((t - t[0]), adc_overvoltage[:, n], label = board_inputs[n])
plt.subplot(211)
plt.ylabel('Overrange')
plt.grid()
plt.legend(fontsize = 10)
plt.subplot(212)
plt.ylabel('Overvoltage')
plt.xlabel('Time Since Start [s]')
plt.grid()
plt.tight_layout()

In [ ]:
plt.figure(figsize = (12, 4))
plt.suptitle('FFT Overflows')
for n in range(N_POLS):
    plt.plot((t - t[0]), fft_overflows[:, n], label = board_inputs[n])
plt.ylabel('Overflows')
plt.xlabel('Time Since Start [s]')
plt.grid()
plt.legend()
plt.tight_layout()

# Check Out All Visibilities

In [ ]:
vis.shape

In [ ]:
# Plot timestreams of all products

f_idx_low = 2040 # 399.22 MHz
f_idx_high = 5200 # 1016.41 MHz
f_idx_gnss = 6200 # 1211.71875, in GNSS range

for n in range(N_PRODS):
    plt.figure(figsize = (12, 3))
    
    p0 = prod[n][0]
    p1 = prod[n][1]
    
    plt.suptitle(f'visibility, product {n}, {board_inputs[p0]}-{board_inputs[p1]}')
    
    plt.subplot(131)
    plt.title(f'{np.round(f[f_idx_low], 2)} MHz')
    plt.plot(t - t[0], np.real(vis[:, f_idx_low, n]), label = 'real')
    if p0 != p1:
        plt.plot(t - t[0], np.imag(vis[:, f_idx_low, n]), label = 'imag')
        plt.legend()
    plt.ylabel('$\mathcal{V}$ [ADU$^2$]')
    plt.xlabel('time since start [s]')
    plt.grid()
    
    plt.subplot(132)
    plt.title(f'{np.round(f[f_idx_high], 2)} MHz')
    plt.plot(t - t[0], np.real(vis[:, f_idx_high, n]))
    if p0 != p1:
        plt.plot(t - t[0], np.imag(vis[:, f_idx_high, n]))
    plt.xlabel('time since start [s]')
    plt.grid()

    plt.subplot(133)
    plt.title(f'{np.round(f[f_idx_gnss], 2)} MHz')
    plt.plot(t - t[0], np.real(vis[:, f_idx_gnss, n]))
    if p0 != p1:
        plt.plot(t - t[0], np.imag(vis[:, f_idx_gnss, n]))
    plt.xlabel('time since start [s]')
    plt.grid()

    plt.tight_layout()

In [ ]:
frame = 0
p = 1
p0 = prod[p][0]
p1 = prod[p][1]

plt.figure(figsize = (12, 6))
plt.title(f'visibility, frame {frame}, {board_inputs[p0]}-{board_inputs[p1]}')
plt.plot(f, np.real(vis[frame, :, p]), label = 'real')
plt.plot(f, np.imag(vis[frame, :, p]), label = 'imag')
plt.xticks(np.arange(0, 1600 + 100, 100))
plt.ylim(-1e7, 1e7)
plt.xlim(0, 1600)
plt.ylabel('$\mathcal{V}$ [ADU$^2$]')
plt.xlabel('frequency [MHz]')
plt.grid()
plt.legend()
plt.tight_layout()

In [ ]:
p = 1
p0 = prod[p][0]
p1 = prod[p][1]

plt.figure(figsize = (12, 6))
plt.title(f'visibility, {board_inputs[p0]}-{board_inputs[p1]}')
plt.pcolormesh(f, t - t[0], np.real(vis[..., p]), vmin = -1e7, vmax = 1e7)
plt.colorbar(label = 'Re($\mathcal{V}$) [ADU$^2$]')
# plt.xticks(np.arange(0, 1600 + 100, 100))
# plt.xlim(0, 150)
plt.ylabel('time [s]')
plt.xlabel('frequency [MHz]')
plt.gca().invert_yaxis()
plt.tight_layout()

In [ ]:
p = 1
p0 = prod[p][0]
p1 = prod[p][1]

plt.figure(figsize = (12, 6))
plt.title(f'visibility, {board_inputs[p0]}-{board_inputs[p1]}')
plt.pcolormesh(f, t - t[0], np.angle(vis[..., p]), vmin = -np.pi, vmax = np.pi, cmap = 'twilight') # 10*np.log10(np.real(vis[..., p])))
plt.colorbar(label = '$\\angle \mathcal{V}$ [rad.]')
plt.xticks(np.arange(0, 1600 + 100, 100))
plt.ylabel('time [s]')
plt.xlabel('frequency [MHz]')
plt.gca().invert_yaxis()
plt.tight_layout()